In [1]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [2]:
from sccnasim.xlib.xdata import load_matrix
from sccnasim.xlib.xrange import format_chrom
from sccnasim.xlib.xvcf import vcf_load

In [3]:
in_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/cellsnp_mode2a'
out_fn = os.path.join(in_dir, 'sccnasim-cs_screadsim.csp_mode2a.h5ad')
anno_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/simu/final/spot_anno.tsv"
phased_snp_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/snp/HCC3N.phased.vcf.gz"

# Load Data

## Raw seed data

In [4]:
def load_phased_snp(fn):
    phase, headers = vcf_load(fn)
    gt_col = phase.columns[-1]
    phase["hap0"] = phase[gt_col].str.split("|").str[0].astype(int)
    phase["hap1"] = phase[gt_col].str.split("|").str[1].astype(int)
    assert np.all(phase["hap0"] + phase["hap1"] == 1)
    phase["chrom"] = phase["CHROM"].map(format_chrom)
    phase["start"] = phase["POS"]
    phase["end"] = phase["POS"]
    phase["snp"] = phase["chrom"] + ":" + phase["POS"].astype(str)
    return(phase)

In [5]:
phase = load_phased_snp(phased_snp_fn)
phase

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,HCC3N,hap0,hap1,chrom,start,end,snp
0,chr1,840279,.,A,G,.,PASS,AD=6;DP=24;OTH=0,GT,1|0,1,0,1,840279,840279,1:840279
1,chr1,996128,.,T,A,.,PASS,AD=48;DP=58;OTH=0,GT,1|0,1,0,1,996128,996128,1:996128
2,chr1,1217251,.,C,A,.,PASS,AD=448;DP=905;OTH=2,GT,0|1,0,1,1,1217251,1217251,1:1217251
3,chr1,1468636,.,G,A,.,PASS,AD=20;DP=45;OTH=1,GT,0|1,0,1,1,1468636,1468636,1:1468636
4,chr1,1719368,.,T,C,.,PASS,AD=16;DP=29;OTH=0,GT,1|0,1,0,1,1719368,1719368,1:1719368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8861,chr9,136862025,.,G,A,.,PASS,AD=9;DP=20;OTH=0,GT,0|1,0,1,9,136862025,136862025,9:136862025
8862,chr9,136954236,.,C,T,.,PASS,AD=36;DP=80;OTH=0,GT,0|1,0,1,9,136954236,136954236,9:136954236
8863,chr9,137273425,.,T,G,.,PASS,AD=435;DP=745;OTH=3,GT,1|0,1,0,9,137273425,137273425,9:137273425
8864,chr9,137273533,.,T,A,.,PASS,AD=171;DP=305;OTH=0,GT,1|0,1,0,9,137273533,137273533,9:137273533


In [6]:
def load_cellsnp(d, cell_anno_fn):
    A = load_matrix(os.path.join(d, "cellSNP.tag.AD.mtx"), "csr")
    D = load_matrix(os.path.join(d, "cellSNP.tag.DP.mtx"), "csr")
    O = load_matrix(os.path.join(d, "cellSNP.tag.OTH.mtx"), "csr")
    
    snps, headers = vcf_load(os.path.join(d, "cellSNP.base.vcf.gz"))
    snps["chrom"] = snps["CHROM"].map(format_chrom)
    snps["start"] = snps["POS"]
    snps["end"] = snps["POS"]
    
    cells = pd.read_table(os.path.join(d, "cellSNP.samples.tsv"), header = None)
    cells.columns = ["cell"]
    
    anno = pd.read_table(cell_anno_fn, header = None)
    anno.columns = ["cell", "cell_type"]
    cells = cells.merge(anno, on = "cell", how = "left")
    
    adata = ad.AnnData(
        X = None,
        obs = snps,
        var = cells
    )
    adata.layers["AD"] = A
    adata.layers["DP"] = D
    adata.layers["OTH"] = O
    adata = adata.transpose().copy()
    
    adata.var["snp"] = adata.var["chrom"] + ":" + adata.var["POS"].astype(str)
    adata = adata[:, ~adata.var["snp"].duplicated(keep = False)].copy()
    return(adata)

In [7]:
adata = load_cellsnp(in_dir, anno_fn)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 6244
    obs: 'cell', 'cell_type'
    var: 'CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'chrom', 'start', 'end', 'snp'
    layers: 'AD', 'DP', 'OTH'

In [8]:
#assert np.all(adata.var['snp'].isin(phase['snp']))

In [9]:
adata.var = adata.var.merge(phase[['hap0', 'hap1', 'snp']], on = 'snp', how = 'left')
adata

AnnData object with n_obs × n_vars = 1200 × 6244
    obs: 'cell', 'cell_type'
    var: 'CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'chrom', 'start', 'end', 'snp', 'hap0', 'hap1'
    layers: 'AD', 'DP', 'OTH'

In [10]:
adata.write_h5ad(out_fn, compression = 'gzip')